<a href="https://colab.research.google.com/github/isocan/ML-accelerated-ORR/blob/main/notebooks/01_stageI_eqv2_adsorbml_relaxation_article_repository.ipynb"
target="_parent"> <img src="https://colab.research.google.com/assets/colab-badge.svg"
    alt="Open in Colab"/> </a>

# Stage I — AdsorbML + EquiformerV2 Relaxation and VASP Export

This repository-ready Google Colab notebook demonstrates the first stage of the oxygen reduction reaction (ORR) screening workflow for the representative material **mp-10260(111)**.

The workflow combines AdsorbML structure generation with EquiformerV2 relaxation, trajectory validation, adsorption-site analysis, minimum-energy structure selection, and repository-ready VASP input generation.

## Notebook Workflow

1. Install the required legacy FAIRChem environment and restart the runtime.
2. Verify the software versions and import the required packages.
3. Download the official Open Catalyst bulk and adsorbate databases.
4. Select `mp-10260` and generate its `(111)` slab terminations.
5. Construct the `O*`, `OH*`, and `OOH*` adsorbates.
6. Generate heuristic adsorption sites together with 20 randomly sampled configurations per adsorbate.
7. Load the `EquiformerV2-31M-S2EF-OC20-All+MD` checkpoint.
8. Relax the constrained bare slab and all adsorbate–slab configurations independently.
9. Validate the resulting relaxation trajectories.
10. Classify the final adsorption sites and remove converged duplicate structures.
11. Visualize the relaxed structures using py3Dmol.
12. Export the summary tables, final structures, and repository-ready VASP input folders.

## Workflow Summary

The notebook performs the following main tasks:

1. Generates and independently relaxes the constrained clean slab using
   `EquiformerV2-31M-S2EF-OC20-All+MD`.

2. Loads `O*` and `OH*` from the Open Catalyst adsorbate database and constructs
   `OOH*` explicitly.

3. Enumerates AdsorbML heuristic atop, bridge, and hollow adsorption sites.

4. Adds 20 randomly sampled adsorption configurations for each adsorbate.

5. Relaxes every adsorbate–slab configuration using EqV2 and the BFGS optimizer until

$$
F_{\max} < 0.05\ \mathrm{eV} \mathring{\mathrm{A}}^{-1},
$$

or until the predefined maximum number of optimization steps is reached.

6. Validates the trajectories, classifies the final adsorption sites, removes
   converged duplicate structures, and identifies the lowest-scoring configurations
   for `O*`, `OH*`, and `OOH*`.

7. Exports the final EqV2 frame of the relaxed bare slab and each adsorbate global
   minimum into separate `vasp/` directories containing:

   * `POSCAR`
   * `INCAR`
   * `KPOINTS`
   * `job.sh`
   * `metadata.json`
   * `POTCAR.spec`
   * a licensing-safe `POTCAR` placeholder

> **Energy interpretation**
>
> EqV2 values are used only to rank the relaxed structures. They are
> machine-learning model scores and must not be interpreted as DFT adsorption
> energies. The independently evaluated bare-slab score is recorded for
> provenance but is not subtracted from the adsorbate–slab scores in this
> notebook.

> **POTCAR licensing**
>
> VASP PAW datasets are not redistributed in this repository. Each exported
> calculation directory contains a deliberately non-runnable `POTCAR`
> placeholder and a `POTCAR.spec` file defining the required elemental order.
> The placeholder must be replaced privately with a properly licensed `POTCAR`
> before running VASP.

## Method Represented in This Notebook

For each slab–adsorbate system,

$$
X^{} \in {\mathrm{O}^{},\ \mathrm{OH}^{},\ \mathrm{OOH}^{}},
$$

the initial adsorption geometries are generated using AdsorbML through a two-step protocol.
protocol.

First, heuristic atop, bridge, and hollow adsorption candidates are enumerated
from the surface geometry. Next, an additional

$$
N_{\mathrm{random}} = 20
$$

configurations are sampled from the surface Delaunay triangulation using
`random_site_heuristic_placement`.

This random-site procedure preserves the designated adsorbate binding atom and
applies a randomized azimuthal orientation together with a small molecular tilt.
It therefore avoids the center-of-mass placement used in the fully random
AdsorbML mode.

The constrained clean slab is relaxed independently using the same EqV2
calculator. All adsorbate configurations are subsequently relaxed using
`EquiformerV2-31M-S2EF-OC20-All+MD` and the BFGS optimizer until

$$
F_{\max} < 0.05\ \mathrm{eV} \mathring{\mathrm{A}}^{-1},
$$

or until the predefined maximum number of optimization steps is reached.

After trajectory validation and duplicate removal, the final VASP package
contains:

* the EqV2-relaxed bare slab;
* the global-minimum `O*` configuration;
* the global-minimum `OH*` configuration; and
* the global-minimum `OOH*` configuration.

In every case, the exported `POSCAR` is generated from the final frame of the
corresponding EqV2 relaxation trajectory.


## 1. Colab environment

In **Runtime → Change runtime type**, choose **T4 GPU** and runtime version
**2026.01**. Check that the version cell reports Python 3.12 before installing
the packages. Colab's 2026.04 and 2026.07 runtimes also use Python 3.12.

The OC20 EquiformerV2 checkpoint uses the legacy FAIRChem v1 stack.
The installer selects PyTorch 2.3.0 with CUDA 12.1 and the matching compiled
PyG extensions. When installation completes, the installer restarts the
kernel automatically. After Colab reconnects, run the verification cell.

[Colab runtime versions](https://research.google.com/colaboratory/runtime-version-faq.html)

In [ ]:
import sys
print(sys.version)
print(sys.executable)

In [ ]:
# ==============================================================
# Google Colab installation for legacy EquiformerV2 / OC20
#
# Colab: use a Python 3.12 runtime (2026.01 recommended).
# Python 3.11 is also supported by the legacy wheels.
#
# Runtime -> Change runtime type
# Runtime Version -> 2026.01
# Hardware accelerator -> GPU
# ==============================================================

from pathlib import Path
import subprocess
import sys
import os

print("=" * 70)
print("EqV2 / FAIRChem legacy environment installer")
print("=" * 70)

print("\nPython executable :", sys.executable)
print("Python version    :", sys.version)

# --------------------------------------------------------------
# IMPORTANT: Check Python version before installing anything
# --------------------------------------------------------------

if sys.version_info[:2] not in {(3, 11), (3, 12)}:
    raise RuntimeError(
        f"Current Python: {sys.version.split()[0]}. "
        "This EqV2 environment requires Python 3.11 or 3.12. "
        "In Colab, select Runtime → Change runtime type, then choose "
        "runtime version 2026.01 and a T4 GPU. Reconnect and run this cell."
    )

print(f"Python {sys.version_info.major}.{sys.version_info.minor} detected.")

# --------------------------------------------------------------
# Marker prevents reinstalling everything after kernel restart
# --------------------------------------------------------------

ENV_MARKER = Path(
    f"/content/.eqv2_legacy_environment_v4_py{sys.version_info.major}{sys.version_info.minor}"
)

if ENV_MARKER.exists():
    print("\n✅ EqV2 environment already installed in this Colab runtime.")
    print("You can continue to the verification/model-loading cells.")

else:

    def run(command):
        print("\n" + "-" * 70)
        print("Running:")
        print(" ".join(command))
        print("-" * 70)
        subprocess.check_call(command)

    # ----------------------------------------------------------
    # 1. Upgrade pip tools
    # ----------------------------------------------------------

    run([
        sys.executable, "-m", "pip", "install",
        "--upgrade",
        "pip",
        "setuptools",
        "wheel",
    ])

    # ----------------------------------------------------------
    # 2. Remove Colab's newer PyTorch stack
    #
    # EqV2 legacy environment will use torch 2.3.0.
    # ----------------------------------------------------------

    subprocess.run([
        sys.executable, "-m", "pip", "uninstall", "-y",
        "torch",
        "torchvision",
        "torchaudio",
    ], check=False)

    # ----------------------------------------------------------
    # 3. Install PyTorch 2.3.0 + CUDA 12.1
    # ----------------------------------------------------------

    run([
        sys.executable, "-m", "pip", "install",
        "--no-cache-dir",
        "torch==2.3.0+cu121",
        "--index-url",
        "https://download.pytorch.org/whl/cu121",
    ])

    # ----------------------------------------------------------
    # 4. Scientific Python stack
    # ----------------------------------------------------------

    run([
        sys.executable, "-m", "pip", "install",
        "--no-cache-dir",
        "numpy==1.26.4",
        "scipy==1.13.1",
        "pandas==2.2.2",
        "ase==3.23.0",
        "pymatgen==2024.10.29",
    ])

    # ----------------------------------------------------------
    # 5. PyTorch Geometric
    # ----------------------------------------------------------

    run([
        sys.executable, "-m", "pip", "install",
        "--no-cache-dir",
        "torch-geometric==2.5.3",
    ])

    # ----------------------------------------------------------
    # 6. PyG compiled extensions
    #
    # Wheels matched to:
    # PyTorch 2.3.0
    # CUDA 12.1
    # ----------------------------------------------------------

    run([
        sys.executable, "-m", "pip", "install",
        "--no-cache-dir",

        "--no-index",
        "--only-binary=:all:",
        "--no-deps",
        "pyg_lib==0.4.0+pt23cu121",
        "torch_scatter==2.1.2+pt23cu121",
        "torch_sparse==0.6.18+pt23cu121",
        "torch_cluster==1.6.3+pt23cu121",
        "torch_spline_conv==1.2.2+pt23cu121",

        "--find-links",
        "https://data.pyg.org/whl/torch-2.3.0+cu121.html",
    ])

    # ----------------------------------------------------------
    # 7. Legacy FAIRChem / Open Catalyst packages
    # ----------------------------------------------------------

    run([
        sys.executable, "-m", "pip", "install",
        "--no-cache-dir",

        "fairchem-core==1.2.0",
        "fairchem-data-oc==0.2.0",

        "py3Dmol>=2.0,<3",
        "ipywidgets>=8,<9",
    ])

    # ----------------------------------------------------------
    # Installation completed
    # ----------------------------------------------------------

    ENV_MARKER.write_text(
        "EqV2 legacy environment installed\n",
        encoding="utf-8"
    )

    print("\n" + "=" * 70)
    print("✅ EqV2 environment installation completed.")
    print("=" * 70)

    print(
        "\nThe Python kernel must now restart because we replaced "
        "Colab's PyTorch installation."
    )

    print("\nRestarting kernel automatically...\n")

    os.kill(os.getpid(), 9)

### Why the kernel must be killed once

`pip` changes the package files on disk, but the current Python process can
still hold the old NumPy, SciPy, or PyTorch modules in memory. That mixed state
caused the earlier `numpy.rec` and version-assertion errors.

`os.kill(os.getpid(), 9)` terminates only the current notebook kernel. Colab
automatically reconnects to a fresh Python process that imports the newly
installed versions. Files under `/content` and the notebook itself are not
deleted. A marker prevents a restart loop when the notebook is run again.


After Colab reconnects, continue with the verification cell below.
The installation marker is specific to the Python version and package setup.

In [ ]:
# Verify the active environment after the restart.

import importlib.metadata as metadata
import sys

import ase
import numpy as np
import scipy
import torch
import torch_geometric
import pyg_lib
import torch_cluster
import torch_scatter
import torch_sparse
import torch_spline_conv

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("PyTorch:", torch.__version__)
print("PyTorch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("ASE:", ase.__version__)
print("torch-geometric:", torch_geometric.__version__)
print("pyg_lib: imported")
print("torch_scatter:", metadata.version("torch-scatter"))
print("torch_sparse:", metadata.version("torch-sparse"))
print("torch_cluster:", metadata.version("torch-cluster"))
print("torch_spline_conv:", metadata.version("torch-spline-conv"))
print("fairchem-core:", metadata.version("fairchem-core"))
print("fairchem-data-oc:", metadata.version("fairchem-data-oc"))

assert np.__version__ == "1.26.4"
assert scipy.__version__ == "1.13.1"
assert torch.__version__ == "2.3.0+cu121"
assert torch.version.cuda == "12.1"
assert metadata.version("torch-scatter").startswith("2.1.2")
assert metadata.version("torch-sparse").startswith("0.6.18")
assert metadata.version("torch-cluster").startswith("1.6.3")
assert metadata.version("torch-spline-conv").startswith("1.2.2")
assert metadata.version("fairchem-core") == "1.2.0"
assert metadata.version("fairchem-data-oc") == "0.2.0"

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. Select Runtime → Change runtime type → T4 GPU."
    )

print("\nEnvironment verification passed.")


In [3]:
# Imports and global settings.

from collections import Counter
from dataclasses import asdict, dataclass

import hashlib
import io
import json
import math
import random
import re
import shutil
import time
import warnings

from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile

import ase.io as aseio
import ipywidgets as widgets
import numpy as np
import pandas as pd
import py3Dmol
import requests
import torch

from ase import Atoms
from ase.calculators.singlepoint import SinglePointCalculator
from ase.constraints import FixAtoms
from ase.data import covalent_radii
from ase.geometry import find_mic
from ase.neighborlist import neighbor_list
from ase.optimize import BFGS
from IPython.display import clear_output, display
from scipy.spatial import ConvexHull, QhullError
from tqdm.auto import tqdm

from fairchem.core.common.relaxation.ase_utils import OCPCalculator
from fairchem.core.models.model_registry import model_name_to_local_file
from fairchem.data.oc.core import Adsorbate, AdsorbateSlabConfig, Bulk, Slab
from fairchem.data.oc.utils import DetectTrajAnomaly

warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

BULK_ID = "mp-10260"
MILLER_INDEX = (1, 1, 1)
SURFACE_INDEX = 0

RANDOM_SITES_PER_ADSORBATE = 20
FMAX_THRESHOLD = 0.05
MAX_RELAX_STEPS = 200
OVERWRITE_EXISTING = False

MODEL_NAME = "EquiformerV2-31M-S2EF-OC20-All+MD"
ADSORBATE_NAMES = ("O", "OH", "OOH")

DATA_DIR = Path("/content/oc_databases")
CHECKPOINT_DIR = Path("/content/fairchem_checkpoints")
ROOT_DIR = Path("/content/stageI_eqv2_results")
SYSTEM_ID = f"{BULK_ID}_{''.join(map(str, MILLER_INDEX))}_term{SURFACE_INDEX}"
SYSTEM_DIR = ROOT_DIR / SYSTEM_ID
TRAJECTORY_DIR = SYSTEM_DIR / "trajectories"
LOG_DIR = SYSTEM_DIR / "logs"
POSCAR_DIR = SYSTEM_DIR / "poscars"
TABLE_DIR = SYSTEM_DIR / "tables"
VASP_DIR = SYSTEM_DIR / "vasp"

KPOINT_DENSITY = 40.0
VASP_RUN_MODE = "single_point"

for directory in (
    DATA_DIR,
    CHECKPOINT_DIR,
    ROOT_DIR,
    SYSTEM_DIR,
    TRAJECTORY_DIR,
    LOG_DIR,
    POSCAR_DIR,
    TABLE_DIR,
    VASP_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)

print("System:", SYSTEM_ID)
print("Random-site placements per adsorbate:", RANDOM_SITES_PER_ADSORBATE)
print("Maximum BFGS steps:", MAX_RELAX_STEPS)
print("Force threshold:", FMAX_THRESHOLD, "eV/Å")


System: mp-10260_111_term0
Random-site placements per adsorbate: 20
Maximum BFGS steps: 200
Force threshold: 0.05 eV/Å


## 2. Download the OC databases

`InDomainBulk.pkl` is not required. The notebook first tries the `bulks.pkl`
file in the archived Open Catalyst Dataset repository supplied for this
workflow. If GitHub returns an HTML page, an LFS pointer, or an incomplete
download, it falls back to the official flat OC bulk database.

O* and OH* are loaded from the official OC adsorbate database. OOH* is
constructed explicitly later because it is not used here as a database lookup.

Only load pickle files from sources you trust. The URLs below are official Open
Catalyst Project resources.


In [4]:
BULK_DB_PATH = DATA_DIR / "bulks.pkl"
ADSORBATE_DB_PATH = DATA_DIR / "adsorbates.pkl"

BULK_URLS = [
    # User-supplied GitHub source.
    "https://github.com/Open-Catalyst-Project/Open-Catalyst-Dataset/"
    "raw/refs/heads/main/ocdata/databases/pkls/bulks.pkl",
    "https://raw.githubusercontent.com/Open-Catalyst-Project/"
    "Open-Catalyst-Dataset/main/ocdata/databases/pkls/bulks.pkl",
    # Official OC20 flat database fallback.
    "https://dl.fbaipublicfiles.com/opencatalystproject/data/"
    "input_generation/bulk_db_flat_2021sep20.pkl",
]

ADSORBATE_URLS = [
    "https://dl.fbaipublicfiles.com/opencatalystproject/data/"
    "input_generation/adsorbate_db_2021apr28.pkl",
]

ADSORBATE_MD5 = "975e00a62c7b634b245102e42167b3fb"
FLAT_BULK_MD5 = "43df744a8366c25392e3072d445016b2"


def md5sum(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.md5()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def invalid_download(path: Path) -> bool:
    if not path.exists() or path.stat().st_size < 1024:
        return True

    with path.open("rb") as handle:
        head = handle.read(512).lower()

    return (
        b"version https://git-lfs.github.com/spec" in head
        or b"<!doctype html" in head
        or b"<html" in head
    )


def download_with_fallback(
    urls: list[str],
    target: Path,
    expected_md5_by_url: dict[str, str] | None = None,
) -> str:
    expected_md5_by_url = expected_md5_by_url or {}

    if target.exists() and not invalid_download(target):
        print(f"Using existing file: {target} ({target.stat().st_size / 1e6:.1f} MB)")
        return "existing"

    target.unlink(missing_ok=True)
    temporary = target.with_suffix(target.suffix + ".part")

    errors = []

    for url in urls:
        temporary.unlink(missing_ok=True)
        print("\nDownloading:", url)

        try:
            with requests.get(url, stream=True, timeout=(30, 300)) as response:
                response.raise_for_status()
                total = int(response.headers.get("content-length", 0))

                with temporary.open("wb") as handle, tqdm(
                    total=total,
                    unit="B",
                    unit_scale=True,
                    desc=target.name,
                ) as progress:
                    for chunk in response.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            handle.write(chunk)
                            progress.update(len(chunk))

            if invalid_download(temporary):
                raise ValueError("The downloaded object is HTML, an LFS pointer, or too small.")

            expected_md5 = expected_md5_by_url.get(url)
            if expected_md5:
                observed_md5 = md5sum(temporary)
                if observed_md5 != expected_md5:
                    raise ValueError(
                        f"MD5 mismatch: expected {expected_md5}, got {observed_md5}"
                    )

            temporary.replace(target)
            print(f"Saved: {target} ({target.stat().st_size / 1e6:.1f} MB)")
            return url

        except Exception as exc:
            errors.append(f"{url}: {type(exc).__name__}: {exc}")
            print("Failed:", errors[-1])

    raise RuntimeError(
        "Every database download failed:\n" + "\n".join(errors)
    )


bulk_source = download_with_fallback(
    BULK_URLS,
    BULK_DB_PATH,
    expected_md5_by_url={
        BULK_URLS[-1]: FLAT_BULK_MD5,
    },
)

adsorbate_source = download_with_fallback(
    ADSORBATE_URLS,
    ADSORBATE_DB_PATH,
    expected_md5_by_url={
        ADSORBATE_URLS[0]: ADSORBATE_MD5,
    },
)

print("\nBulk source:", bulk_source)
print("Adsorbate source:", adsorbate_source)



Downloading: https://github.com/Open-Catalyst-Project/Open-Catalyst-Dataset/raw/refs/heads/main/ocdata/databases/pkls/bulks.pkl


bulks.pkl:   0%|          | 0.00/36.1M [00:00<?, ?B/s]

Saved: /content/oc_databases/bulks.pkl (36.1 MB)

Downloading: https://dl.fbaipublicfiles.com/opencatalystproject/data/input_generation/adsorbate_db_2021apr28.pkl


adsorbates.pkl:   0%|          | 0.00/57.0k [00:00<?, ?B/s]

Saved: /content/oc_databases/adsorbates.pkl (0.1 MB)

Bulk source: https://github.com/Open-Catalyst-Project/Open-Catalyst-Dataset/raw/refs/heads/main/ocdata/databases/pkls/bulks.pkl
Adsorbate source: https://dl.fbaipublicfiles.com/opencatalystproject/data/input_generation/adsorbate_db_2021apr28.pkl


In [22]:
# ==============================================================
# Bulk visualization + downloadable 1x1x1 CIF
#
# Visualization:
#     centered 3x3x3 crystal (-1, 0, +1)
#
# Saved/downloaded CIF:
#     original 1x1x1 unit cell only
# ==============================================================

from pathlib import Path
import io
import itertools

import ase.io as aseio
import py3Dmol
import ipywidgets as widgets
from IPython.display import display


# --------------------------------------------------------------
# Output directory
# --------------------------------------------------------------

BULK_STRUCTURE_DIR = SYSTEM_DIR / "bulk_structure"
BULK_STRUCTURE_DIR.mkdir(parents=True, exist_ok=True)

UNITCELL_CIF_PATH = (
    BULK_STRUCTURE_DIR /
    f"{BULK_ID}_1x1x1_unitcell.cif"
)


# ==============================================================
# 1. Original 1x1x1 bulk unit cell
# ==============================================================

primitive = bulk_atoms.copy()

primitive.set_pbc(
    (True, True, True)
)

# Put all atoms consistently inside the unit cell
primitive.wrap()


# --------------------------------------------------------------
# Structural information
# --------------------------------------------------------------

a, b, c = primitive.cell.lengths()
alpha, beta, gamma = primitive.cell.angles()

print("=" * 72)
print(f"Bulk structure: {BULK_ID}")
print("=" * 72)

print(f"Formula          : {primitive.get_chemical_formula()}")
print(f"Atoms/unit cell  : {len(primitive)}")
print(f"Volume           : {primitive.get_volume():.3f} Å³")

print("\nLattice parameters")
print("-" * 40)

print(f"a      = {a:10.5f} Å")
print(f"b      = {b:10.5f} Å")
print(f"c      = {c:10.5f} Å")

print(f"alpha  = {alpha:10.4f}°")
print(f"beta   = {beta:10.4f}°")
print(f"gamma  = {gamma:10.4f}°")


# ==============================================================
# 2. Save ONLY the original 1x1x1 unit cell as CIF
# ==============================================================

aseio.write(
    str(UNITCELL_CIF_PATH),
    primitive,
    format="cif",
)

print("\n1x1x1 CIF saved:")
print(UNITCELL_CIF_PATH)


# ==============================================================
# 3. Build centered 3x3x3 structure ONLY for visualization
#
# lattice translations:
#
#        -1, 0, +1
#
# along a, b and c
# ==============================================================

cell = primitive.cell.array

a_vec = cell[0]
b_vec = cell[1]
c_vec = cell[2]

translations = (-1, 0, 1)

images = []

for i, j, k in itertools.product(
    translations,
    translations,
    translations,
):

    image = primitive.copy()

    translation_vector = (
        i * a_vec
        + j * b_vec
        + k * c_vec
    )

    image.translate(
        translation_vector
    )

    images.append(image)


# --------------------------------------------------------------
# Combine all 27 cells
# --------------------------------------------------------------

visual_atoms = images[0].copy()

for image in images[1:]:
    visual_atoms += image


# Explicit coordinates are already present,
# so periodic boundary conditions are unnecessary for display.
visual_atoms.set_pbc(False)


print("\nVisualization:")
print("Periodic range       : -1 ... +1")
print("Equivalent structure : 3 × 3 × 3")
print(f"Displayed atoms      : {len(visual_atoms)}")

expected_atoms = len(primitive) * 27

print(f"Expected atoms       : {expected_atoms}")

assert len(visual_atoms) == expected_atoms


# ==============================================================
# 4. Convert visualization structure to XYZ
#
# XYZ is used only internally for py3Dmol.
# Nothing is saved as XYZ.
# ==============================================================

xyz_buffer = io.StringIO()

aseio.write(
    xyz_buffer,
    visual_atoms,
    format="xyz",
)

xyz_text = xyz_buffer.getvalue()


# ==============================================================
# 5. Interactive py3Dmol visualization
# ==============================================================

view = py3Dmol.view(
    width=850,
    height=620,
)

view.addModel(
    xyz_text,
    "xyz",
)


# --------------------------------------------------------------
# Nickel
# --------------------------------------------------------------

view.setStyle(
    {"elem": "Ni"},
    {
        "sphere": {
            "radius": 0.52,
            "color": "#43A047",
        },
        "stick": {
            "radius": 0.10,
            "color": "#43A047",
        },
    },
)


# --------------------------------------------------------------
# Antimony
# --------------------------------------------------------------

view.setStyle(
    {"elem": "Sb"},
    {
        "sphere": {
            "radius": 0.72,
            "color": "#7E57C2",
        },
        "stick": {
            "radius": 0.11,
            "color": "#7E57C2",
        },
    },
)


view.setBackgroundColor("white")

view.zoomTo()

# Slightly zoom out so boundary atoms remain visible
view.zoom(0.85)

view.show()


# ==============================================================
# 6. Google Colab download button
#
# IMPORTANT:
# This downloads ONLY the original 1x1x1 unit-cell CIF.
# ==============================================================

download_button = widgets.Button(
    description="Download 1×1×1 CIF",
    button_style="success",
    icon="download",
)


def download_unitcell_cif(_):

    from google.colab import files

    files.download(
        str(UNITCELL_CIF_PATH)
    )


download_button.on_click(
    download_unitcell_cif
)


print("\nDownloadable structure:")
print("Original 1×1×1 unit cell")
print(UNITCELL_CIF_PATH)

display(download_button)

Bulk structure: mp-10260
Formula          : Ni3Sb
Atoms/unit cell  : 4
Volume           : 53.642 Å³

Lattice parameters
----------------------------------------
a      =    4.23324 Å
b      =    4.23324 Å
c      =    4.23324 Å
alpha  =    60.0000°
beta   =    60.0000°
gamma  =    60.0000°

1x1x1 CIF saved:
/content/stageI_eqv2_results/mp-10260_111_term0/bulk_structure/mp-10260_1x1x1_unitcell.cif

Visualization:
Periodic range       : -1 ... +1
Equivalent structure : 3 × 3 × 3
Displayed atoms      : 108
Expected atoms       : 108


3Dmol.js failed to load for some reason. Please check your browser console for error messages.


Downloadable structure:
Original 1×1×1 unit cell
/content/stageI_eqv2_results/mp-10260_111_term0/bulk_structure/mp-10260_1x1x1_unitcell.cif


Button(button_style='success', description='Download 1×1×1 CIF', icon='download', style=ButtonStyle())

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 3. Generate the initial `mp-10260(111)` slab

The bulk is selected by Materials Project identifier, not by database position.
The code explicitly checks the returned ID because the legacy `Bulk` class can
otherwise fall back to a random material when an ID is absent.

The generated slab is saved before relaxation for provenance. After the EqV2
checkpoint is loaded, the same constrained slab is relaxed independently and
its final trajectory frame is exported as the repository `vasp/bare/POSCAR`.
The adsorbate placement step remains based on the reproducibly generated initial
slab, matching the AdsorbML screening protocol represented here.


Support for third party widgets will remain active for the duration of the session. To disable support:

In [5]:
from google.colab import output
output.disable_custom_widget_manager()

In [6]:
def rebuild_atoms(
    atoms: Atoms,
    *,
    constrain_subsurface: bool = False,
    molecule: bool = False,
) -> Atoms:
    # Recreate an ASE object to avoid incompatibilities from old pickled ASE objects.
    tags = np.asarray(atoms.get_tags(), dtype=int)

    rebuilt = Atoms(
        numbers=np.asarray(atoms.get_atomic_numbers(), dtype=int),
        positions=np.asarray(atoms.get_positions(), dtype=float),
        cell=np.asarray(atoms.cell.array, dtype=float),
        pbc=False if molecule else np.asarray(atoms.get_pbc(), dtype=bool),
        tags=tags,
    )

    if constrain_subsurface:
        fixed_indices = np.flatnonzero(tags == 0)
        if len(fixed_indices):
            rebuilt.set_constraint(FixAtoms(indices=fixed_indices))

    return rebuilt


def ensure_finite_geometry(atoms: Atoms, label: str) -> None:
    if not np.isfinite(atoms.positions).all():
        raise ValueError(f"Non-finite positions in {label}.")
    if not np.isfinite(atoms.cell.array).all():
        raise ValueError(f"Non-finite cell in {label}.")


bulk = Bulk(
    bulk_src_id_from_db=BULK_ID,
    bulk_db_path=str(BULK_DB_PATH),
)

if bulk.src_id != BULK_ID:
    raise RuntimeError(
        f"Requested {BULK_ID}, but the database returned {bulk.src_id}. "
        "The requested material is missing from this database."
    )

bulk.atoms = rebuild_atoms(bulk.atoms)

generated_slabs = Slab.from_bulk_get_specific_millers(
    bulk=bulk,
    specific_millers=MILLER_INDEX,
)

slab_candidates = (
    [generated_slabs]
    if isinstance(generated_slabs, Slab)
    else list(generated_slabs)
)

if not slab_candidates:
    raise RuntimeError(f"No {MILLER_INDEX} slab was generated for {BULK_ID}.")

if not 0 <= SURFACE_INDEX < len(slab_candidates):
    raise IndexError(
        f"SURFACE_INDEX={SURFACE_INDEX} but only "
        f"{len(slab_candidates)} terminations were generated."
    )

raw_slab_object = slab_candidates[SURFACE_INDEX]
clean_slab = rebuild_atoms(
    raw_slab_object.atoms,
    constrain_subsurface=True,
)
ensure_finite_geometry(clean_slab, "clean slab")

# Rebuild the Slab wrapper around the current ASE object.
slab_object = Slab(
    bulk=raw_slab_object.bulk,
    slab_atoms=clean_slab,
    millers=raw_slab_object.millers,
    shift=raw_slab_object.shift,
    top=raw_slab_object.top,
    oriented_bulk=raw_slab_object.oriented_bulk,
)

clean_traj_path = SYSTEM_DIR / "clean_slab_unrelaxed.traj"
clean_poscar_path = POSCAR_DIR / "clean_slab_unrelaxed" / "POSCAR"
clean_poscar_path.parent.mkdir(parents=True, exist_ok=True)

aseio.write(clean_traj_path, clean_slab)
aseio.write(
    clean_poscar_path,
    clean_slab,
    format="vasp",
    direct=True,
    vasp5=True,
    sort=True,
)

tag_counts = (
    pd.Series(clean_slab.get_tags(), name="count")
    .value_counts()
    .sort_index()
    .rename_axis("OC_tag")
    .reset_index(name="count")
)

print("Bulk ID:", bulk.src_id)
print("Bulk formula:", bulk.atoms.get_chemical_formula())
print("Requested Miller index:", MILLER_INDEX)
print("Generated terminations:", len(slab_candidates))
print("Selected termination:", SURFACE_INDEX)
print("Slab formula:", clean_slab.get_chemical_formula())
print("Number of slab atoms:", len(clean_slab))
print("Saved clean slab:", clean_poscar_path)
display(tag_counts)


Bulk ID: mp-10260
Bulk formula: Ni3Sb
Requested Miller index: (1, 1, 1)
Generated terminations: 4
Selected termination: 0
Slab formula: Ni36Sb12
Number of slab atoms: 48
Saved clean slab: /content/stageI_eqv2_results/mp-10260_111_term0/poscars/clean_slab_unrelaxed/POSCAR


,OC_tag,count
0,0,36
1,1,12


## 4. Define O*, OH*, and OOH*

- O* is loaded from the OC adsorbate database using `*O`.
- OH* is loaded from the OC adsorbate database using `*OH`.
- OOH* is constructed with the binding O at index 0.

The OOH coordinates use an O–O distance of 1.45 Å and an approximately
0.99 Å terminal O–H bond. AdsorbML binds the first oxygen to the sampled site.


In [7]:
def database_adsorbate(smiles: str) -> Adsorbate:
    adsorbate = Adsorbate(
        adsorbate_smiles_from_db=smiles,
        adsorbate_db_path=str(ADSORBATE_DB_PATH),
    )
    if adsorbate.smiles != smiles:
        raise RuntimeError(
            f"Requested adsorbate {smiles}, but database returned {adsorbate.smiles}."
        )
    adsorbate.atoms = rebuild_atoms(adsorbate.atoms, molecule=True)
    return adsorbate


adsorbate_O = database_adsorbate("*O")
adsorbate_OH = database_adsorbate("*OH")

ooh_atoms = Atoms(
    symbols=["O", "O", "H"],
    positions=[
        [0.000, 0.000, 0.000],  # binding O
        [0.000, 0.000, 1.450],  # distal O
        [0.940, 0.000, 1.750],  # terminal H
    ],
    pbc=False,
)

adsorbate_OOH = Adsorbate(
    adsorbate_atoms=ooh_atoms,
    adsorbate_binding_indices=[0],
)

adsorbates = {
    "O": adsorbate_O,
    "OH": adsorbate_OH,
    "OOH": adsorbate_OOH,
}

adsorbate_summary = pd.DataFrame(
    [
        {
            "adsorbate": name,
            "source": "OC adsorbate database" if name in {"O", "OH"} else "explicit coordinates",
            "smiles": getattr(adsorbate, "smiles", None),
            "formula": adsorbate.atoms.get_chemical_formula(),
            "binding_indices": list(adsorbate.binding_indices),
            "n_atoms": len(adsorbate.atoms),
        }
        for name, adsorbate in adsorbates.items()
    ]
)

display(adsorbate_summary)


,adsorbate,source,smiles,formula,binding_indices,n_atoms
0,O,OC adsorbate database,*O,O,[0],1
1,OH,OC adsorbate database,*OH,HO,[0],2
2,OOH,explicit coordinates,None,HO2,[0],3


## 5. Generate AdsorbML initial structures

For each adsorbate, this cell keeps:

- every heuristic placement returned by AdsorbML;
- exactly 20 additional random surface sites generated with
  `random_site_heuristic_placement`.

No heuristic placements are truncated.


In [8]:
placement_records = []
configuration_records = []

for adsorbate_name in ADSORBATE_NAMES:
    adsorbate = adsorbates[adsorbate_name]

    heuristic_generator = AdsorbateSlabConfig(
        slab_object,
        adsorbate,
        mode="heuristic",
    )

    random_generator = AdsorbateSlabConfig(
        slab_object,
        adsorbate,
        mode="random_site_heuristic_placement",
        num_sites=RANDOM_SITES_PER_ADSORBATE,
    )

    groups = [
        (
            "heuristic",
            list(heuristic_generator.atoms_list),
            list(heuristic_generator.metadata_list),
        ),
        (
            "random_site",
            list(random_generator.atoms_list),
            list(random_generator.metadata_list),
        ),
    ]

    running_index = 0

    for placement_kind, atoms_list, metadata_list in groups:
        if len(atoms_list) != len(metadata_list):
            raise RuntimeError(
                f"Atoms/metadata mismatch for {adsorbate_name} {placement_kind}."
            )

        for local_index, (atoms, metadata_item) in enumerate(
            zip(atoms_list, metadata_list)
        ):
            rebuilt = rebuild_atoms(
                atoms,
                constrain_subsurface=True,
            )
            ensure_finite_geometry(
                rebuilt,
                f"{adsorbate_name} {placement_kind} {local_index}",
            )

            config_id = f"{adsorbate_name}_{running_index:03d}"

            site = np.asarray(metadata_item.get("site", [np.nan] * 3), dtype=float)
            xyz_angles = metadata_item.get("xyz_angles", None)

            configuration_records.append(
                {
                    "config_id": config_id,
                    "adsorbate": adsorbate_name,
                    "config_index": running_index,
                    "placement_kind": placement_kind,
                    "placement_local_index": local_index,
                    "initial_site_x_A": float(site[0]),
                    "initial_site_y_A": float(site[1]),
                    "initial_site_z_A": float(site[2]),
                    "sampled_angles": repr(xyz_angles),
                    "atoms": rebuilt,
                }
            )
            running_index += 1

        placement_records.append(
            {
                "adsorbate": adsorbate_name,
                "placement_kind": placement_kind,
                "n_configurations": len(atoms_list),
            }
        )

placements = pd.DataFrame(placement_records)
configuration_table = pd.DataFrame(
    [{key: value for key, value in record.items() if key != "atoms"}
     for record in configuration_records]
)

placement_summary = (
    placements
    .pivot(index="adsorbate", columns="placement_kind", values="n_configurations")
    .fillna(0)
    .astype(int)
    .reset_index()
)

placement_summary["total"] = (
    placement_summary.get("heuristic", 0)
    + placement_summary.get("random_site", 0)
)

display(placement_summary)
display(configuration_table.head())

assert (placement_summary["random_site"] == RANDOM_SITES_PER_ADSORBATE).all()


placement_kind,adsorbate,heuristic,random_site,total
0,O,7,20,27
1,OH,7,20,27
2,OOH,7,20,27


,config_id,adsorbate,config_index,placement_kind,placement_local_index,initial_site_x_A,initial_site_y_A,initial_site_z_A,sampled_angles
0,O_000,O,0,heuristic,0,1.641920e-15,2.444064e+00,18.303848,"array([0, 0, 0])"
1,O_001,O,1,heuristic,1,2.116622e+00,1.222032e+00,19.167955,"array([0, 0, 0])"
2,O_002,O,2,heuristic,2,1.058311e+00,6.110161e-01,19.600009,"array([0, 0, 0])"
3,O_003,O,3,heuristic,3,1.619421e-15,8.208959e-16,20.032062,"array([0, 0, 0])"
4,O_004,O,4,heuristic,4,2.116622e+00,3.256148e-15,18.735902,"array([0, 0, 0])"


## 6. Load the EquiformerV2 checkpoint

The model is downloaded once and cached under `/content/fairchem_checkpoints`.
The calculator predicts OC20 adsorption-referenced model scores and forces.


In [9]:
checkpoint_path = model_name_to_local_file(
    MODEL_NAME,
    local_cache=str(CHECKPOINT_DIR),
)

print("Checkpoint:", checkpoint_path)

try:
    calculator = OCPCalculator(
        checkpoint_path=checkpoint_path,
        cpu=False,
        seed=SEED,
    )
except TypeError:
    # Compatibility fallback for minor constructor differences.
    calculator = OCPCalculator(
        checkpoint_path=checkpoint_path,
        cpu=False,
    )

print("EqV2 calculator loaded.")


Checkpoint: /content/fairchem_checkpoints/eq2_31M_ec4_allmd.pt


EqV2 calculator loaded.


/usr/local/lib/python3.11/dist-packages/fairchem/core/modules/normalization/normalizer.py:69: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "mean": torch.tensor(state_dict["mean"]),


## 7. Relax the bare slab and every adsorbate–slab configuration

The clean slab is relaxed independently with EqV2 and saved as
`trajectories/bare/bare_slab_eqv2.traj`. Each adsorbate trajectory is also
independent and is written to its own `.traj` file. Failed calculations are
recorded rather than stopping the complete screening run. Existing trajectories
can be reused by leaving `OVERWRITE_EXISTING=False`.

A structure is considered converged when the final maximum atomic force is
strictly below 0.05 eV Å\(^{-1}\). The bare-slab EqV2 score is retained only as
run metadata; the adsorbate configurations are ranked directly by their EqV2
scores.


In [10]:
def maximum_force_from_array(forces: np.ndarray) -> float:
    forces = np.asarray(forces, dtype=float)
    if forces.size == 0:
        return float("nan")
    return float(np.linalg.norm(forces, axis=1).max())


def final_frame_with_results(atoms: Atoms) -> Atoms:
    """Copy a final geometry and preserve its energy and forces."""
    energy = float(atoms.get_potential_energy())
    forces = np.asarray(atoms.get_forces(), dtype=float)

    final = atoms.copy()
    final.calc = SinglePointCalculator(
        final,
        energy=energy,
        forces=forces,
    )
    return final


def read_frames(path: Path) -> list[Atoms]:
    frames = aseio.read(str(path), index=":")
    return frames if isinstance(frames, list) else [frames]


def write_complete_trajectory(
    trajectory_path: Path,
    frames: list[Atoms],
    final: Atoms,
) -> list[Atoms]:
    """Ensure traj[-1] exactly matches the returned relaxed structure."""
    if len(frames) == 0:
        frames = [final]
    elif np.max(np.abs(frames[-1].positions - final.positions)) > 1e-10:
        frames.append(final)
    else:
        frames[-1] = final

    aseio.write(
        str(trajectory_path),
        frames,
        format="traj",
    )
    return frames


def relax_bare_slab() -> dict:
    bare_traj_dir = TRAJECTORY_DIR / "bare"
    bare_log_dir = LOG_DIR / "bare"
    bare_traj_dir.mkdir(parents=True, exist_ok=True)
    bare_log_dir.mkdir(parents=True, exist_ok=True)

    trajectory_path = bare_traj_dir / "bare_slab_eqv2.traj"
    log_path = bare_log_dir / "bare_slab_eqv2.log"

    result = {
        "system_id": SYSTEM_ID,
        "structure": "bare",
        "trajectory": str(trajectory_path),
        "log": str(log_path),
        "relaxation_status": "failed",
        "error": "",
        "elapsed_s": np.nan,
        "n_frames": 0,
        "ml_score_eV": np.nan,
        "final_fmax_eV_A": np.nan,
        "converged": False,
        "final_poscar": "",
    }

    if trajectory_path.exists() and not OVERWRITE_EXISTING:
        try:
            frames = read_frames(trajectory_path)
            final = frames[-1]
            energy = float(final.get_potential_energy())
            forces = np.asarray(final.get_forces(), dtype=float)
            fmax = maximum_force_from_array(forces)

            final_poscar_dir = POSCAR_DIR / "bare_eqv2_relaxed"
            final_poscar_dir.mkdir(parents=True, exist_ok=True)
            final_poscar_path = final_poscar_dir / "POSCAR"
            aseio.write(
                final_poscar_path,
                final,
                format="vasp",
                direct=True,
                vasp5=True,
                sort=True,
            )

            return {
                **result,
                "relaxation_status": "reused",
                "n_frames": len(frames),
                "ml_score_eV": energy,
                "final_fmax_eV_A": fmax,
                "converged": bool(
                    np.isfinite(fmax)
                    and fmax < FMAX_THRESHOLD
                ),
                "final_poscar": str(final_poscar_path),
            }
        except Exception:
            trajectory_path.unlink(missing_ok=True)

    atoms = rebuild_atoms(
        clean_slab,
        constrain_subsurface=True,
    )
    atoms.calc = calculator

    start = time.time()

    try:
        optimizer = BFGS(
            atoms,
            trajectory=str(trajectory_path),
            logfile=str(log_path),
        )
        optimizer.run(
            fmax=FMAX_THRESHOLD,
            steps=MAX_RELAX_STEPS,
        )

        final = final_frame_with_results(atoms)
        frames = read_frames(trajectory_path)
        frames = write_complete_trajectory(
            trajectory_path,
            frames,
            final,
        )

        energy = float(final.get_potential_energy())
        forces = np.asarray(final.get_forces(), dtype=float)
        fmax = maximum_force_from_array(forces)

        final_poscar_dir = POSCAR_DIR / "bare_eqv2_relaxed"
        final_poscar_dir.mkdir(parents=True, exist_ok=True)
        final_poscar_path = final_poscar_dir / "POSCAR"
        aseio.write(
            final_poscar_path,
            final,
            format="vasp",
            direct=True,
            vasp5=True,
            sort=True,
        )

        return {
            **result,
            "relaxation_status": "completed",
            "elapsed_s": time.time() - start,
            "n_frames": len(frames),
            "ml_score_eV": energy,
            "final_fmax_eV_A": fmax,
            "converged": bool(
                np.isfinite(fmax)
                and fmax < FMAX_THRESHOLD
            ),
            "final_poscar": str(final_poscar_path),
        }

    except Exception as exc:
        return {
            **result,
            "elapsed_s": time.time() - start,
            "error": f"{type(exc).__name__}: {exc}",
        }


bare_slab_status = relax_bare_slab()

print("Bare-slab EqV2 relaxation")
display(pd.DataFrame([bare_slab_status]))


def relax_one_configuration(record: dict) -> dict:
    adsorbate_name = record["adsorbate"]
    config_id = record["config_id"]

    adsorbate_traj_dir = TRAJECTORY_DIR / adsorbate_name
    adsorbate_log_dir = LOG_DIR / adsorbate_name
    adsorbate_traj_dir.mkdir(parents=True, exist_ok=True)
    adsorbate_log_dir.mkdir(parents=True, exist_ok=True)

    trajectory_path = adsorbate_traj_dir / f"{config_id}.traj"
    log_path = adsorbate_log_dir / f"{config_id}.log"

    base_result = {
        "system_id": SYSTEM_ID,
        "config_id": config_id,
        "adsorbate": adsorbate_name,
        "config_index": int(record["config_index"]),
        "placement_kind": record["placement_kind"],
        "trajectory": str(trajectory_path),
        "log": str(log_path),
        "relaxation_status": "failed",
        "error": "",
        "elapsed_s": np.nan,
        "n_frames": 0,
        "ml_score_eV": np.nan,
        "final_fmax_eV_A": np.nan,
        "converged": False,
    }

    if trajectory_path.exists() and not OVERWRITE_EXISTING:
        try:
            frames = read_frames(trajectory_path)
            final = frames[-1]
            energy = float(final.get_potential_energy())
            forces = np.asarray(final.get_forces(), dtype=float)
            fmax = maximum_force_from_array(forces)

            return {
                **base_result,
                "relaxation_status": "reused",
                "n_frames": len(frames),
                "ml_score_eV": energy,
                "final_fmax_eV_A": fmax,
                "converged": bool(
                    np.isfinite(fmax)
                    and fmax < FMAX_THRESHOLD
                ),
            }
        except Exception:
            trajectory_path.unlink(missing_ok=True)

    atoms = rebuild_atoms(
        record["atoms"],
        constrain_subsurface=True,
    )
    atoms.calc = calculator

    start = time.time()

    try:
        optimizer = BFGS(
            atoms,
            trajectory=str(trajectory_path),
            logfile=str(log_path),
        )
        optimizer.run(
            fmax=FMAX_THRESHOLD,
            steps=MAX_RELAX_STEPS,
        )

        final = final_frame_with_results(atoms)
        frames = read_frames(trajectory_path)
        frames = write_complete_trajectory(
            trajectory_path,
            frames,
            final,
        )

        energy = float(final.get_potential_energy())
        forces = np.asarray(final.get_forces(), dtype=float)
        fmax = maximum_force_from_array(forces)

        return {
            **base_result,
            "relaxation_status": "completed",
            "elapsed_s": time.time() - start,
            "n_frames": len(frames),
            "ml_score_eV": energy,
            "final_fmax_eV_A": fmax,
            "converged": bool(
                np.isfinite(fmax)
                and fmax < FMAX_THRESHOLD
            ),
        }

    except Exception as exc:
        return {
            **base_result,
            "elapsed_s": time.time() - start,
            "error": f"{type(exc).__name__}: {exc}",
        }


relaxation_rows = []

for record in tqdm(
    configuration_records,
    desc="EqV2 adsorbate relaxations",
):
    relaxation_rows.append(relax_one_configuration(record))

relaxation_status = pd.DataFrame(relaxation_rows)

display(
    relaxation_status[
        [
            "adsorbate",
            "config_id",
            "placement_kind",
            "relaxation_status",
            "converged",
            "ml_score_eV",
            "final_fmax_eV_A",
            "elapsed_s",
            "error",
        ]
    ]
)

print("\nAdsorbate relaxation summary")
display(
    relaxation_status.groupby(
        ["adsorbate", "relaxation_status", "converged"],
        dropna=False,
    ).size().rename("count").reset_index()
)


Bare-slab EqV2 relaxation


,system_id,structure,trajectory,log,relaxation_status,error,elapsed_s,n_frames,ml_score_eV,final_fmax_eV_A,converged,final_poscar
0,mp-10260_111_term0,bare,/content/stageI_eqv2_results/mp-10260_111_term...,/content/stageI_eqv2_results/mp-10260_111_term...,completed,,1.785071,7,-0.754395,0.018848,True,/content/stageI_eqv2_results/mp-10260_111_term...


EqV2 adsorbate relaxations:   0%|          | 0/81 [00:00<?, ?it/s]

,adsorbate,config_id,placement_kind,relaxation_status,converged,ml_score_eV,final_fmax_eV_A,elapsed_s,error
0,O,O_000,heuristic,completed,True,2.638672,0.035097,1.688478,
1,O,O_001,heuristic,completed,True,2.304688,0.032868,1.682374,
2,O,O_002,heuristic,completed,True,1.826172,0.046999,5.972501,
3,O,O_003,heuristic,completed,True,2.181641,0.046976,2.467123,
4,O,O_004,heuristic,completed,True,1.830078,0.040386,7.863900,
...,...,...,...,...,...,...,...,...,...
76,OOH,OOH_022,random_site,completed,True,3.876953,0.036061,11.913675,
77,OOH,OOH_023,random_site,completed,True,3.888672,0.049747,9.355634,
78,OOH,OOH_024,random_site,completed,True,2.503906,0.032662,9.252376,
79,OOH,OOH_025,random_site,completed,True,3.888672,0.031171,13.059230,



Adsorbate relaxation summary


,adsorbate,relaxation_status,converged,count
0,O,completed,True,27
1,OH,completed,True,27
2,OOH,completed,True,27


## 8. Validate trajectories and export every final frame

The validator uses the OC tag convention:

- tag 0: fixed/subsurface slab atom;
- tag 1: surface slab atom;
- tag 2: adsorbate atom.

This avoids the unsafe element-based rule that would incorrectly classify
oxygen or hydrogen atoms belonging to an oxide or hydroxylated slab.

Every readable final frame, accepted or rejected, is exported to an individual
VASP `POSCAR`. A trajectory is accepted for site analysis when it is converged, finite,
and passes the FAIRChem anomaly checks. The adsorbate composition, slab
atom order, and cell are verified before those checks.


In [ ]:
ADSORBATE_FORMULAS = {
    "O": Counter(O=1),
    "OH": Counter(O=1, H=1),
    "OOH": Counter(O=2, H=1),
}


def identify_adsorbate(atoms: Atoms, name: str, bare: Atoms) -> tuple[np.ndarray, np.ndarray, str]:
    """Use OC tag 2, with a checked appended-atom fallback for untagged frames."""
    expected = ADSORBATE_FORMULAS[name]
    ids = np.flatnonzero(atoms.get_tags() == 2)
    source = "tags_2"
    if not len(ids):
        if len(atoms) != len(bare) + sum(expected.values()):
            raise ValueError("Cannot identify an appended adsorbate from the bare slab and atom count.")
        ids = np.arange(len(bare), len(atoms))
        source = "validated_appended_atoms"
    if Counter(atoms[i].symbol for i in ids) != expected:
        raise ValueError(f"Adsorbate atom tags or appended atoms do not match {name}.")
    slab = np.setdiff1d(np.arange(len(atoms)), ids)
    if not np.array_equal(atoms.numbers[slab], bare.numbers):
        raise ValueError("Trajectory and bare slab differ in atom order or composition.")
    if not np.allclose(atoms.cell, bare.cell, rtol=0, atol=1e-5):
        raise ValueError("Trajectory and bare slab have different cells.")
    return ids, slab, source


def anomaly_flags(frames: list[Atoms], adsorbate_name: str) -> dict:
    initial = frames[0].copy()
    final = frames[-1].copy()

    initial_ads, initial_slab, source = identify_adsorbate(initial, adsorbate_name, clean_slab)
    final_ads, _, _ = identify_adsorbate(final, adsorbate_name, clean_slab)
    if not np.array_equal(initial_ads, final_ads):
        raise ValueError("Adsorbate atom indices changed between trajectory endpoints.")

    initial_tags = initial.get_tags()
    if source == "validated_appended_atoms":
        initial_tags[initial_slab] = clean_slab.get_tags()
        initial_tags[initial_ads] = 2

    detector = DetectTrajAnomaly(
        initial,
        final,
        initial_tags,
    )

    return {
        "adsorbate_dissociated": bool(detector.is_adsorbate_dissociated()),
        "adsorbate_desorbed": bool(detector.is_adsorbate_desorbed()),
        "surface_changed": bool(detector.has_surface_changed()),
        "adsorbate_intercalated": bool(detector.is_adsorbate_intercalated()),
    }


def write_poscar(atoms: Atoms, directory: Path) -> Path:
    directory.mkdir(parents=True, exist_ok=True)
    poscar_path = directory / "POSCAR"
    aseio.write(
        poscar_path,
        atoms,
        format="vasp",
        direct=True,
        vasp5=True,
        sort=True,
    )
    return poscar_path


scan_rows = []

for row in relaxation_status.itertuples(index=False):
    record = {
        "system_id": row.system_id,
        "config_id": row.config_id,
        "adsorbate": row.adsorbate,
        "config_index": int(row.config_index),
        "placement_kind": row.placement_kind,
        "trajectory": row.trajectory,
        "relaxation_status": row.relaxation_status,
        "validation_status": "rejected",
        "rejection_reason": "",
        "ml_score_eV": row.ml_score_eV,
        "final_fmax_eV_A": row.final_fmax_eV_A,
        "n_frames": int(row.n_frames),
        "final_poscar": "",
        "adsorbate_dissociated": False,
        "adsorbate_desorbed": False,
        "surface_changed": False,
        "adsorbate_intercalated": False,
    }

    if row.relaxation_status not in {"completed", "reused"}:
        record["rejection_reason"] = row.error or "relaxation_failed"
        scan_rows.append(record)
        continue

    try:
        frames = read_frames(Path(row.trajectory))
        final = frames[-1]

        all_final_dir = (
            POSCAR_DIR
            / "all_final_frames"
            / row.adsorbate
            / row.config_id
        )
        record["final_poscar"] = str(write_poscar(final, all_final_dir))

        flags = anomaly_flags(frames, row.adsorbate)
        record.update(flags)

        active_anomalies = [name for name, value in flags.items() if value]

        if not np.isfinite(row.ml_score_eV):
            record["rejection_reason"] = "non_finite_ml_score"
        elif not np.isfinite(row.final_fmax_eV_A):
            record["rejection_reason"] = "non_finite_force"
        elif row.final_fmax_eV_A >= FMAX_THRESHOLD:
            record["rejection_reason"] = (
                f"force_above_threshold:{row.final_fmax_eV_A:.6f}"
            )
        elif active_anomalies:
            record["rejection_reason"] = ";".join(active_anomalies)
        else:
            record["validation_status"] = "accepted"

    except Exception as exc:
        record["rejection_reason"] = (
            f"validation_failed:{type(exc).__name__}:{exc}"
        )

    scan_rows.append(record)

trajectory_scan = pd.DataFrame(scan_rows)

display(
    trajectory_scan[
        [
            "adsorbate",
            "config_id",
            "placement_kind",
            "validation_status",
            "rejection_reason",
            "ml_score_eV",
            "final_fmax_eV_A",
            "final_poscar",
        ]
    ]
)

print("\nValidation summary")
display(
    trajectory_scan.groupby(
        ["adsorbate", "validation_status"],
        dropna=False,
    ).size().rename("count").reset_index()
)


## 9. Classify final adsorption sites

Site labels describe the final relaxed frame. The detector uses the first
shell of oxygen–slab contacts in three dimensions, with periodic images
along the two in-plane lattice vectors. Contact distances are normalized
by sums of covalent radii. The geometry of the first shell distinguishes
top, bridge, threefold, fourfold, and noncanonical motifs.

For OOH*, the oxygen farther from H defines the primary motif. Contacts
from the H-bearing oxygen are retained to distinguish binding through
one or both oxygen atoms. The confidence flag identifies assignments that
are cutoff-sensitive, noncanonical, or involve the H-bearing oxygen.

- A **site instance** is identified by its contact atom indices and
  relative periodic image offsets.
- A **site family** groups motifs with the same neighboring elements,
  including secondary oxygen contacts when present.
- The **global minimum** for each adsorbate is the lowest-scoring accepted
  EqV2 configuration. Confidence flags are recorded alongside the score.

Contact-free assignments are retained in the accepted-configuration table
and omitted from the site-instance and site-family counts. The exported
tables include the periodic offsets, contact distances, and assignment
confidence.

In [ ]:
# Adapted from the common detector in the OC20 and AQCat trajectory analyses.
SITE_DETECTOR_VERSION = "periodic-first-shell-1.1"


@dataclass(frozen=True)
class SiteSettings:
    candidate_cutoff_A: float = 4.5
    contact_ratio: float = 1.25
    shell_gap: float = 0.15
    sensitivity: float = 0.04
    bridge_projection_A: float = 0.65


SITE_SETTINGS = SiteSettings()


def contact_candidates(
    atoms: Atoms,
    oxygens: list[int],
    slab: np.ndarray,
    settings: SiteSettings,
) -> dict:
    """Collect O–slab contacts and retain the lattice image of each neighbor."""
    a, b = atoms.cell.array[:2]
    e1 = a / np.linalg.norm(a)
    normal = np.cross(a, b)
    normal /= np.linalg.norm(normal)
    e2 = np.cross(normal, e1)
    slab_ids = set(slab)
    candidates = {i: [] for i in oxygens}
    ii, jj, shifts, vectors, distances = neighbor_list(
        "ijSDd", atoms, settings.candidate_cutoff_A, self_interaction=False
    )
    for i, j, shift, vector, distance in zip(ii, jj, shifts, vectors, distances):
        if i not in candidates or j not in slab_ids:
            continue
        reference = float(covalent_radii[atoms.numbers[i]] + covalent_radii[atoms.numbers[j]])
        candidates[i].append({
            "index": int(j),
            "symbol": atoms[j].symbol,
            "image": shift.tolist(),
            "distance_A": float(distance),
            "reference_length_A": reference,
            "ratio": float(distance / reference),
            "xy": [float(vector @ e1), float(vector @ e2)],
        })
    for contacts in candidates.values():
        contacts.sort(key=lambda contact: (contact["ratio"], contact["index"], contact["image"]))
    return candidates


def select_contacts(candidates: list[dict], settings: SiteSettings, delta: float = 0.0) -> list[dict]:
    if not candidates:
        return []
    limit = min(settings.contact_ratio, candidates[0]["ratio"] + settings.shell_gap) + delta
    return [contact for contact in candidates if contact["ratio"] <= limit]


def geometry_label(contacts: list[dict], settings: SiteSettings) -> tuple[str, float | None, bool]:
    """Classify the geometry of the selected first-shell contacts."""
    n = len(contacts)
    if n == 0:
        return "unbound", None, False
    xy = np.asarray([contact["xy"] for contact in contacts])
    if n == 1:
        return "top", float(np.linalg.norm(xy[0])), True
    if n == 2:
        direction = xy[1] - xy[0]
        norm_squared = direction @ direction
        if norm_squared < 1e-10:
            return "irregular_2coord", None, False
        t = float(-xy[0] @ direction / norm_squared)
        residual = float(np.linalg.norm(xy[0] + t * direction))
        bridge = 0.10 <= t <= 0.90 and residual <= settings.bridge_projection_A
        return ("bridge" if bridge else "offcenter_2coord"), residual, bridge
    if n in (3, 4):
        try:
            hull = ConvexHull(xy)
            regular = len(hull.vertices) == n and bool(np.all(hull.equations[:, -1] <= 1e-7))
        except QhullError:
            regular = False
        if regular:
            return f"{n}-fold", None, True
    return f"irregular_{n}coord", None, False


def canonical_edges(per_oxygen: list[list[dict]]) -> tuple:
    """Use contact offsets relative to a common lattice origin."""
    edges = [(role, contact["index"], *contact["image"])
             for role, contacts in enumerate(per_oxygen) for contact in contacts]
    return min(
        (tuple(sorted((role, index, sx - x, sy - y, sz - z)
                      for role, index, sx, sy, sz in edges))
         for _, _, x, y, z in edges),
        default=(),
    )


def detect_adsorption_site(
    atoms: Atoms,
    adsorbate_name: str,
    bare: Atoms,
    settings: SiteSettings = SITE_SETTINGS,
) -> dict:
    adsorbate, slab, identity_source = identify_adsorbate(atoms, adsorbate_name, bare)
    geometry = atoms.copy()
    geometry.pbc = (True, True, False)
    geometry.wrap(eps=1e-8)
    # Anchor each slab atom to its bare-slab image across relaxed configurations.
    displacement, _ = find_mic(
        geometry.positions[slab] - bare.positions, geometry.cell, geometry.pbc
    )
    geometry.positions[slab] = bare.positions + displacement
    oxygens = [int(i) for i in adsorbate if geometry[i].symbol == "O"]
    if adsorbate_name == "OOH":
        hydrogen = next(i for i in adsorbate if geometry[i].symbol == "H")
        _, distances = find_mic(
            geometry.positions[oxygens] - geometry.positions[hydrogen],
            geometry.cell, geometry.pbc,
        )
        distal = oxygens.pop(int(np.argmin(distances)))
        anchor = oxygens[0]
        displacement, _ = find_mic(
            geometry.positions[distal] - geometry.positions[anchor], geometry.cell, geometry.pbc
        )
        geometry.positions[distal] = geometry.positions[anchor] + displacement
        oxygens.append(distal)

    candidates = contact_candidates(geometry, oxygens, slab, settings)
    per_oxygen = [select_contacts(candidates[o], settings) for o in oxygens]
    selected = sorted(per_oxygen[0], key=lambda contact: (contact["index"], contact["image"]))
    label, residual, regular = geometry_label(selected, settings)
    edges = canonical_edges(per_oxygen)
    stable = all(
        canonical_edges([select_contacts(candidates[o], settings, delta) for o in oxygens]) == edges
        for delta in (-settings.sensitivity, settings.sensitivity)
    )

    distal_contacts = per_oxygen[1] if adsorbate_name == "OOH" else []
    notes = []
    if not stable:
        notes.append("cutoff_sensitive_contacts")
    if not regular:
        notes.append("noncanonical_or_unbound_geometry")
    if distal_contacts:
        notes.append("multiple_oxygen_surface_contacts" if selected else "distal_only_binding")

    composition = "-".join(sorted(contact["symbol"] for contact in selected))
    family = f"{label}__{composition}"
    if distal_contacts:
        family += "__other_O_" + "-".join(sorted(contact["symbol"] for contact in distal_contacts))
    if distal_contacts:
        binding_mode = "multi_oxygen" if selected else "distal_only"
    else:
        binding_mode = "single_oxygen" if selected else "unbound"

    def compact_json(value):
        return json.dumps(value, separators=(",", ":"))

    distances = [contact["distance_A"] for contact in selected]
    return {
        "site_type": label,
        "coordination": len(selected),
        "binding_oxygen_index": oxygens[0],
        "binding_mode": binding_mode,
        "adsorbate_identity_source": identity_source,
        "neighbor_composition": composition,
        "neighbor_indices": "-".join(str(contact["index"]) for contact in selected),
        "neighbor_images": compact_json([contact["image"] for contact in selected]),
        "neighbor_distances_A": ";".join(f"{distance:.4f}" for distance in distances),
        "site_instance_key": f"{label}__edges_{compact_json(edges)}",
        "site_family": family,
        "local_site_descriptor": f"{family}__d_" + "-".join(f"{d:.2f}" for d in sorted(distances)),
        "site_assignment_reason": ";".join(notes) or "3D_first_shell_contacts",
        "site_confidence": "review" if notes else "clear",
        "site_stable_to_cutoff": stable,
        "projection_residual_A": residual,
        "oxygen_contacts_json": compact_json(dict(zip(oxygens, per_oxygen))),
    }

In [ ]:
accepted = trajectory_scan.loc[
    trajectory_scan["validation_status"] == "accepted"
].copy()

site_rows = []
for row in accepted.itertuples(index=False):
    final = aseio.read(row.trajectory, index=-1)
    site_rows.append(detect_adsorption_site(final, row.adsorbate, clean_slab))

all_adsorption_sites = pd.concat(
    [accepted.reset_index(drop=True), pd.DataFrame(site_rows)], axis=1
)

if all_adsorption_sites.empty:
    unique_site_minima = all_adsorption_sites.copy()
    site_family_minima = all_adsorption_sites.copy()
    global_minima = all_adsorption_sites.copy()
    print("No accepted trajectory is available for site analysis.")
else:
    system_columns = ["system_id", "adsorbate"]
    energy_column = "ml_score_eV"
    all_adsorption_sites["delta_from_adsorbate_min_eV"] = (
        all_adsorption_sites[energy_column]
        - all_adsorption_sites.groupby(system_columns)[energy_column].transform("min")
    )
    by_energy = all_adsorption_sites.sort_values(
        [*system_columns, energy_column, "config_index"]
    )
    # Keep unassigned configurations in the energy analysis without counting a
    # contact-free geometry as a distinct adsorption site.
    assigned = by_energy.loc[by_energy["binding_mode"] != "unbound"]
    unique_site_minima = assigned.drop_duplicates(
        [*system_columns, "site_instance_key"]
    ).reset_index(drop=True)
    site_family_minima = assigned.drop_duplicates(
        [*system_columns, "site_family"]
    ).reset_index(drop=True)
    global_minima = by_energy.drop_duplicates(system_columns).reset_index(drop=True)

    print("Site assignments by adsorbate and confidence")
    display(all_adsorption_sites.groupby(["adsorbate", "site_confidence"]).size().unstack(fill_value=0))

    site_columns = [
        "adsorbate", "config_id", "site_family", "site_confidence",
        "binding_mode", "ml_score_eV",
    ]
    print(f"Unique site instances ({len(unique_site_minima)})")
    display(unique_site_minima[site_columns])
    print(f"Site families ({len(site_family_minima)})")
    display(site_family_minima[site_columns])
    print("Global minimum per adsorbate")
    display(global_minima[[*site_columns, "site_assignment_reason"]])

## 10. Interactive py3Dmol visualization

The selector below uses one py3Dmol output area. It contains the initial clean
slab, the independently EqV2-relaxed bare slab, every global adsorbate minimum,
and every unique-site minimum. Selecting a new entry replaces the previous
molecular viewer rather than creating many static plots.


In [ ]:
# Interactive py3Dmol structure viewer.

try:
    from google.colab import output

    output.enable_custom_widget_manager()
except Exception:
    pass


def atoms_to_cif_string(atoms: Atoms) -> str:
    """Convert an ASE Atoms object to a CIF-formatted string."""
    buffer = io.BytesIO()
    aseio.write(buffer, atoms, format="cif")
    return buffer.getvalue().decode("utf-8")


def show_atoms_py3dmol(
    atoms: Atoms,
    *,
    width: int = 850,
    height: int = 520,
) -> None:
    """Display an ASE Atoms object using py3Dmol."""
    viewer = py3Dmol.view(
        width=width,
        height=height,
    )
    viewer.addModel(
        atoms_to_cif_string(atoms),
        "cif",
    )
    viewer.setStyle(
        {
            "sphere": {"scale": 0.32},
            "stick": {"radius": 0.14},
        }
    )
    viewer.addUnitCell()
    viewer.zoomTo()
    viewer.show()


structure_catalog = {
    "Clean slab | initial": str(clean_traj_path),
}

bare_trajectory_path = Path(
    bare_slab_status.get("trajectory", "")
)
if bare_trajectory_path.exists():
    structure_catalog[
        "Bare slab | EqV2-relaxed"
    ] = str(bare_trajectory_path)


if not global_minima.empty:
    for row in global_minima.itertuples(index=False):
        label = (
            f"GLOBAL MIN | {row.adsorbate}* | "
            f"{row.site_family} | "
            f"{row.ml_score_eV:.4f} eV | "
            f"{row.config_id}"
        )
        structure_catalog[label] = str(row.trajectory)


if not unique_site_minima.empty:
    for row in unique_site_minima.itertuples(index=False):
        label = (
            f"UNIQUE SITE | {row.adsorbate}* | "
            f"{row.site_family} | "
            f"{row.ml_score_eV:.4f} eV | "
            f"{row.config_id}"
        )
        structure_catalog[label] = str(row.trajectory)


default_structure = (
    "Bare slab | EqV2-relaxed"
    if "Bare slab | EqV2-relaxed" in structure_catalog
    else "Clean slab | initial"
)

selector = widgets.Dropdown(
    options=list(structure_catalog.keys()),
    value=default_structure,
    description="Structure:",
    layout=widgets.Layout(width="95%"),
)

viewer_output = widgets.Output()


def render_selected_structure(change=None) -> None:
    """Load and display the structure selected in the dropdown."""
    label = selector.value
    trajectory_path = Path(structure_catalog[label])

    with viewer_output:
        clear_output(wait=True)

        if not trajectory_path.exists():
            print(f"Structure file not found: {trajectory_path}")
            return

        try:
            atoms = aseio.read(
                str(trajectory_path),
                index=-1,
            )

            print(label)
            show_atoms_py3dmol(atoms)

        except Exception as exc:
            print(f"Could not display {label}")
            print(f"{type(exc).__name__}: {exc}")


selector.observe(
    render_selected_structure,
    names="value",
)

display(
    widgets.VBox(
        [
            selector,
            viewer_output,
        ]
    )
)

render_selected_structure()


## 11. Export tables, selected structures, and article-repository VASP inputs

Every readable adsorbate trajectory already has a `POSCAR` under
`poscars/all_final_frames/`. This final export additionally creates:

- the lowest-scoring structure for each explicit unique site;
- the global EqV2 minimum for O*, OH*, and OOH*;
- `vasp/bare`, `vasp/O`, `vasp/OH`, and `vasp/OOH`;
- `POSCAR`, `INCAR`, `KPOINTS`, `job.sh`, `metadata.json`, `POTCAR.spec`,
  and a non-distributable `POTCAR` placeholder in each VASP folder;
- `vasp/manifest.csv` and `vasp/README.md`;
- CSV summaries for placements, relaxations, validation, and site analysis;
- one ZIP archive containing the complete result directory.

The supplied INCAR represents a VASP **single-point validation** calculation on
the final EqV2 geometry (`NSW = 0`). Replace the placeholder POTCAR privately
before running VASP.


In [ ]:
def export_selected_poscars(
    table: pd.DataFrame,
    category: str,
) -> pd.DataFrame:
    exported = []

    if table.empty:
        return pd.DataFrame()

    for row in table.itertuples(index=False):
        final = aseio.read(
            row.trajectory,
            index=-1,
        )

        if category == "unique_site_minima":
            key_digest = hashlib.sha256(row.site_instance_key.encode()).hexdigest()[:12]
            folder_name = f"{row.config_id}__{key_digest}"
        else:
            folder_name = (
                f"{row.adsorbate}_global_min__"
                f"{row.config_id}"
            )

        target_directory = (
            POSCAR_DIR
            / category
            / row.adsorbate
            / folder_name
        )
        poscar_path = write_poscar(
            final,
            target_directory,
        )

        exported.append(
            {
                "category": category,
                "adsorbate": row.adsorbate,
                "config_id": row.config_id,
                "site_type": row.site_type,
                "site_family": row.site_family,
                "site_instance_key": row.site_instance_key,
                "site_confidence": row.site_confidence,
                "binding_mode": row.binding_mode,
                "ml_score_eV": row.ml_score_eV,
                "trajectory": row.trajectory,
                "poscar": str(poscar_path),
            }
        )

    return pd.DataFrame(exported)


unique_poscars = export_selected_poscars(
    unique_site_minima,
    "unique_site_minima",
)
global_poscars = export_selected_poscars(
    global_minima,
    "global_minima",
)
selected_poscars = pd.concat(
    [unique_poscars, global_poscars],
    ignore_index=True,
)


def write_poscar_with_selective_dynamics(
    poscar_path: Path,
    atoms: Atoms,
) -> list[str]:
    """Write a deterministic VASP5 POSCAR and preserve FixAtoms flags."""
    symbols = atoms.get_chemical_symbols()

    element_order = []
    for symbol in symbols:
        if symbol not in element_order:
            element_order.append(symbol)

    grouped_indices = [
        index
        for element in element_order
        for index, symbol in enumerate(symbols)
        if symbol == element
    ]
    counts = [
        sum(symbol == element for symbol in symbols)
        for element in element_order
    ]

    fixed_indices = set()
    for constraint in getattr(
        atoms,
        "constraints",
        [],
    ):
        if isinstance(constraint, FixAtoms):
            fixed_indices.update(
                map(int, constraint.get_indices())
            )

    scaled_positions = atoms.get_scaled_positions(
        wrap=False,
    )
    cell = np.asarray(
        atoms.cell.array,
        dtype=float,
    )

    lines = [
        f"{SYSTEM_ID} final EqV2 frame",
        "1.0",
    ]
    lines.extend(
        "  "
        + " ".join(
            f"{component:.12f}"
            for component in vector
        )
        for vector in cell
    )
    lines.append(
        "  " + " ".join(element_order)
    )
    lines.append(
        "  " + " ".join(map(str, counts))
    )
    lines.append("Selective dynamics")
    lines.append("Direct")

    for index in grouped_indices:
        position = scaled_positions[index]
        flags = (
            "F F F"
            if index in fixed_indices
            else "T T T"
        )
        lines.append(
            "  "
            + " ".join(
                f"{component:.12f}"
                for component in position
            )
            + f"  {flags}"
        )

    poscar_path.write_text(
        "\n".join(lines) + "\n",
        encoding="utf-8",
    )
    return element_order


def write_kpoints(
    directory: Path,
    atoms: Atoms,
) -> tuple[int, int, int]:
    cell = np.asarray(
        atoms.cell.array,
        dtype=float,
    )
    a_length = float(np.linalg.norm(cell[0]))
    b_length = float(np.linalg.norm(cell[1]))

    if a_length <= 0.0 or b_length <= 0.0:
        raise ValueError(
            "Invalid in-plane lattice vectors for KPOINTS."
        )

    kx = max(
        1,
        int(round(KPOINT_DENSITY / a_length)),
    )
    ky = max(
        1,
        int(round(KPOINT_DENSITY / b_length)),
    )
    kz = 1

    text = (
        "Automatic mesh\n"
        "0\n"
        "Monkhorst-Pack\n"
        f"{kx} {ky} {kz}\n"
        "0 0 0\n"
    )
    (directory / "KPOINTS").write_text(
        text,
        encoding="utf-8",
    )
    return kx, ky, kz


def write_incar(
    directory: Path,
    system_label: str,
) -> None:
    text = f"""SYSTEM = {system_label}
PREC = Accurate
ENCUT = 350
GGA = RP
EDIFF = 1E-5
NELM = 120

# Single-point evaluation of the final EqV2 geometry
IBRION = -1
NSW = 0
ISIF = 2

ISYM = 0
SYMPREC = 1E-10
LREAL = Auto
LASPH = .TRUE.
LWAVE = .FALSE.
LCHARG = .FALSE.
NCORE = 4
"""
    (directory / "INCAR").write_text(
        text,
        encoding="utf-8",
    )


def safe_job_name(label: str) -> str:
    cleaned = re.sub(
        r"[^A-Za-z0-9_-]+",
        "-",
        label,
    ).strip("-")
    return cleaned[:64] or "vasp_SP"


def write_batch_script(
    directory: Path,
    job_label: str,
) -> None:
    job_name = safe_job_name(job_label)
    text = f"""#!/bin/bash
#SBATCH -N 1
#SBATCH -n 64
#SBATCH --time=00:59:00
#SBATCH -p rome
#SBATCH -J {job_name}
#SBATCH --output=out.%j
#SBATCH --error=err.%j

set -euo pipefail

echo "SLURM_JOBID ${{SLURM_JOBID:-unset}}"
echo "SLURM_NODELIST ${{SLURM_NODELIST:-unset}}"

if grep -q "POTCAR NOT DISTRIBUTED" POTCAR; then
    echo "ERROR: Replace the repository POTCAR placeholder with a licensed POTCAR."
    exit 2
fi

module purge
module load 2024
module load VASP6/6.4.3-foss-2024a

srun vasp_std > vasp.out
"""
    script_path = directory / "job.sh"
    script_path.write_text(
        text,
        encoding="utf-8",
    )
    script_path.chmod(0o755)


def write_potcar_placeholder(
    directory: Path,
    elements: list[str],
) -> None:
    element_line = " ".join(elements)

    placeholder = f"""POTCAR NOT DISTRIBUTED

This is an intentional repository placeholder, not a VASP potential file.
Required POSCAR/POTCAR element order:
{element_line}

Create the real POTCAR privately from your licensed VASP PAW-PBE library.
Do not commit or redistribute the licensed POTCAR.
"""
    (directory / "POTCAR").write_text(
        placeholder,
        encoding="utf-8",
    )

    spec = (
        "# Required POTCAR order; one element per line.\n"
        + "\n".join(elements)
        + "\n"
    )
    (directory / "POTCAR.spec").write_text(
        spec,
        encoding="utf-8",
    )


def export_vasp_case(
    case_name: str,
    trajectory_path: str,
    metadata: dict,
) -> dict:
    source_path = Path(trajectory_path)

    if not source_path.exists():
        raise FileNotFoundError(
            f"Trajectory not found for {case_name}: "
            f"{source_path}"
        )

    atoms = aseio.read(
        str(source_path),
        index=-1,
    )

    run_directory = VASP_DIR / case_name
    if run_directory.exists():
        shutil.rmtree(run_directory)
    run_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    poscar_path = run_directory / "POSCAR"
    elements = write_poscar_with_selective_dynamics(
        poscar_path,
        atoms,
    )
    kmesh = write_kpoints(
        run_directory,
        atoms,
    )
    write_incar(
        run_directory,
        f"{SYSTEM_ID}_{case_name}_EqV2_final",
    )
    write_batch_script(
        run_directory,
        f"{SYSTEM_ID}_{case_name}_SP",
    )
    write_potcar_placeholder(
        run_directory,
        elements,
    )

    case_metadata = {
        "system_id": SYSTEM_ID,
        "case": case_name,
        "source_trajectory": str(source_path),
        "source_frame": -1,
        "formula": atoms.get_chemical_formula(),
        "n_atoms": len(atoms),
        "element_order": elements,
        "kpoint_mesh": list(kmesh),
        "kpoint_density_parameter": KPOINT_DENSITY,
        "vasp_run_mode": VASP_RUN_MODE,
        "potcar_distributed": False,
        **metadata,
    }

    with (run_directory / "metadata.json").open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            case_metadata,
            handle,
            indent=2,
        )

    return {
        "case": case_name,
        "directory": str(run_directory),
        "poscar": str(poscar_path),
        "trajectory": str(source_path),
        "formula": atoms.get_chemical_formula(),
        "n_atoms": len(atoms),
        "element_order": " ".join(elements),
        "kpoints": " ".join(map(str, kmesh)),
        "ml_score_eV": metadata.get(
            "ml_score_eV",
            np.nan,
        ),
        "config_id": metadata.get(
            "config_id",
            "",
        ),
        "site_family": metadata.get(
            "site_family",
            "",
        ),
    }


vasp_records = []

bare_trajectory = Path(
    bare_slab_status.get("trajectory", "")
)
if (
    bare_slab_status.get("relaxation_status")
    in {"completed", "reused"}
    and bare_trajectory.exists()
):
    vasp_records.append(
        export_vasp_case(
            "bare",
            str(bare_trajectory),
            {
                "structure_type": "bare_slab",
                "relaxation_status": bare_slab_status[
                    "relaxation_status"
                ],
                "converged": bool(
                    bare_slab_status["converged"]
                ),
                "ml_score_eV": float(
                    bare_slab_status["ml_score_eV"]
                ),
                "final_fmax_eV_A": float(
                    bare_slab_status["final_fmax_eV_A"]
                ),
            },
        )
    )
else:
    print(
        "[WARN] Bare slab was not exported to VASP "
        "because no readable completed trajectory exists."
    )


available_global_adsorbates = set(
    global_minima.get(
        "adsorbate",
        pd.Series(dtype=str),
    ).tolist()
)
missing_global_adsorbates = (
    set(ADSORBATE_NAMES)
    - available_global_adsorbates
)
if missing_global_adsorbates:
    print(
        "[WARN] No accepted global minimum for:",
        ", ".join(sorted(missing_global_adsorbates)),
    )


for adsorbate_name in ADSORBATE_NAMES:
    if (
        global_minima.empty
        or "adsorbate" not in global_minima.columns
    ):
        selected = pd.DataFrame()
    else:
        selected = global_minima[
            global_minima["adsorbate"]
            == adsorbate_name
        ]

    if selected.empty:
        continue

    row = selected.iloc[0]
    vasp_records.append(
        export_vasp_case(
            adsorbate_name,
            str(row["trajectory"]),
            {
                "structure_type": "adsorbate_global_minimum",
                "adsorbate": adsorbate_name,
                "config_id": str(row["config_id"]),
                "placement_kind": str(
                    row["placement_kind"]
                ),
                "site_type": str(row["site_type"]),
                "site_family": str(row["site_family"]),
                "site_instance_key": str(
                    row["site_instance_key"]
                ),
                "neighbor_composition": str(
                    row["neighbor_composition"]
                ),
                "neighbor_images": json.loads(row["neighbor_images"]),
                "binding_mode": str(row["binding_mode"]),
                "site_confidence": str(row["site_confidence"]),
                "site_assignment_reason": str(row["site_assignment_reason"]),
                "site_detector_version": SITE_DETECTOR_VERSION,
                "ml_score_eV": float(
                    row["ml_score_eV"]
                ),
                "final_fmax_eV_A": float(
                    row["final_fmax_eV_A"]
                ),
            },
        )
    )


vasp_manifest = pd.DataFrame(
    vasp_records,
)

readme_text = f"""# VASP inputs from final EqV2 structures

System: `{SYSTEM_ID}`

This directory contains one folder for the independently EqV2-relaxed bare
slab and one folder for each accepted adsorbate global minimum:

- `bare/`
- `O/`
- `OH/`
- `OOH/`

Each folder contains:

- `POSCAR`: final frame (`traj[-1]`) from the selected EqV2 trajectory;
- `INCAR`: RPBE single-point settings (`NSW = 0`);
- `KPOINTS`: Monkhorst-Pack mesh generated with
  `round({KPOINT_DENSITY}/|a|) × round({KPOINT_DENSITY}/|b|) × 1`;
- `job.sh`: example DIFFER/Snellius-style Slurm script;
- `metadata.json`: source trajectory, model score, force, and site metadata;
- `POTCAR.spec`: required element order;
- `POTCAR`: deliberately invalid licensing placeholder.

## Important POTCAR notice

VASP PAW datasets are licensed and are not included in this repository.
Before running VASP, replace each placeholder `POTCAR` privately with the
licensed PAW-PBE datasets in exactly the order listed in `POTCAR.spec`.
Never commit or redistribute the licensed replacement file.

The included `job.sh` checks for the placeholder text and exits before launching
VASP when the real POTCAR has not been supplied.
"""
(VASP_DIR / "README.md").write_text(
    readme_text,
    encoding="utf-8",
)
vasp_manifest.to_csv(
    VASP_DIR / "manifest.csv",
    index=False,
)


tables = {
    "adsorbate_summary.csv": adsorbate_summary,
    "placement_summary.csv": placement_summary,
    "initial_configuration_metadata.csv": configuration_table,
    "bare_slab_relaxation.csv": pd.DataFrame(
        [bare_slab_status]
    ),
    "relaxation_status.csv": relaxation_status,
    "trajectory_validation.csv": trajectory_scan,
    "all_adsorption_sites.csv": all_adsorption_sites,
    "unique_site_minima.csv": unique_site_minima,
    "site_family_minima.csv": site_family_minima,
    "global_minima.csv": global_minima,
    "selected_poscars.csv": selected_poscars,
    "vasp_manifest.csv": vasp_manifest,
}

for filename, table in tables.items():
    table.to_csv(
        TABLE_DIR / filename,
        index=False,
    )


run_metadata = {
    "bulk_id": BULK_ID,
    "miller_index": MILLER_INDEX,
    "surface_index": SURFACE_INDEX,
    "adsorption_site_detection": {
        "version": SITE_DETECTOR_VERSION,
        "periodicity": [True, True, False],
        "settings": asdict(SITE_SETTINGS),
    },
    "bulk_formula": bulk.atoms.get_chemical_formula(),
    "slab_formula": clean_slab.get_chemical_formula(),
    "model_name": MODEL_NAME,
    "random_sites_per_adsorbate": RANDOM_SITES_PER_ADSORBATE,
    "fmax_threshold_eV_A": FMAX_THRESHOLD,
    "max_relax_steps": MAX_RELAX_STEPS,
    "bare_slab_relaxed": bool(
        bare_slab_status.get(
            "relaxation_status"
        ) in {"completed", "reused"}
    ),
    "bare_slab_converged": bool(
        bare_slab_status.get(
            "converged",
            False,
        )
    ),
    "bare_slab_ml_score_eV": (
        float(bare_slab_status["ml_score_eV"])
        if np.isfinite(
            bare_slab_status.get(
                "ml_score_eV",
                np.nan,
            )
        )
        else None
    ),
    "bare_slab_energy_used_for_adsorption_energy": False,
    "vasp_run_mode": VASP_RUN_MODE,
    "vasp_cases_exported": vasp_manifest[
        "case"
    ].tolist() if not vasp_manifest.empty else [],
    "potcar_distributed": False,
    "o_source": "OC adsorbate database: *O",
    "oh_source": "OC adsorbate database: *OH",
    "ooh_source": (
        "explicit O-O-H coordinates; "
        "binding atom index 0"
    ),
}

with (SYSTEM_DIR / "run_metadata.json").open(
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        run_metadata,
        handle,
        indent=2,
    )


archive_path = ROOT_DIR / f"{SYSTEM_ID}_stageI_eqv2_repository.zip"
selection_folders = (POSCAR_DIR / "unique_site_minima", POSCAR_DIR / "global_minima")
current_poscars = {Path(path) for path in selected_poscars.get("poscar", [])}
exported_cases = {VASP_DIR / case for case in vasp_manifest.get("case", [])}
stale_cases = {VASP_DIR / case for case in ("bare", *ADSORBATE_NAMES)} - exported_cases

with ZipFile(archive_path, "w", compression=ZIP_DEFLATED) as archive:
    for path in sorted(SYSTEM_DIR.rglob("*")):
        if not path.is_file() or any(path.is_relative_to(folder) for folder in stale_cases):
            continue
        if any(path.is_relative_to(folder) for folder in selection_folders) and path not in current_poscars:
            continue
        archive.write(path, path.relative_to(ROOT_DIR))

print("Result directory:", SYSTEM_DIR)
print("ZIP archive:", archive_path)
print("\nVASP cases:")
display(vasp_manifest)

print("\nTop-level outputs:")
for path in sorted(SYSTEM_DIR.iterdir()):
    print(" -", path.name)


In [ ]:
# Optional Colab download.

from pathlib import Path

archive_path = Path(archive_path)

try:
    from google.colab import files

    files.download(str(archive_path))
except ImportError:
    print("Archive available at:", archive_path)

## Output interpretation

- `tables/bare_slab_relaxation.csv` records the independent EqV2 relaxation of
  the constrained clean slab.
- `tables/trajectory_validation.csv` contains every attempted adsorbate
  configuration and the reason for rejection, where applicable.
- `tables/all_adsorption_sites.csv` contains every accepted final adsorbate
  configuration.
- `tables/unique_site_minima.csv` removes trajectories that converged to the
  same periodic oxygen–slab contact graph and retains the lowest EqV2 score.
- `tables/site_family_minima.csv` groups sites by coordination and neighboring
  elements.
- `tables/global_minima.csv` contains one lowest-scoring structure for O*, OH*,
  and OOH*.
- `poscars/all_final_frames/` contains `traj[-1] → POSCAR` exports for every
  readable adsorbate trajectory.
- `poscars/bare_eqv2_relaxed/` contains the final EqV2-relaxed bare slab.
- `poscars/unique_site_minima/` and `poscars/global_minima/` contain the
  structures selected during screening.
- `vasp/bare/`, `vasp/O/`, `vasp/OH/`, and `vasp/OOH/` contain the final
  repository-ready VASP input packages.
- `tables/selected_poscars.csv` maps each exported structure to its full site key.
- `vasp/manifest.csv` records the source trajectory and selection metadata for
  each exported VASP folder.

The VASP inputs are intentionally non-runnable until the placeholder `POTCAR`
in each folder is replaced privately with licensed PAW-PBE datasets in the
order specified by `POTCAR.spec`.


If the trajectory files and relaxation results are still available in the
Colab session, rerun Sections 8–11 to regenerate the site tables and exports.

## References

1. Lan, J.; Palizhati, A.; Shuaibi, M.; *et al.*  
   **AdsorbML: A Leap in Efficiency for Adsorption Energy Calculations Using
   Generalizable Machine Learning Potentials.**  
   *npj Computational Materials* **2023**.

2. Liao, Y.-L.; Wood, B. M.; Das, A.; Smidt, T.  
   **EquiformerV2: Improved Equivariant Transformer for Scaling to
   Higher-Degree Representations.**  
   *ICLR* **2024**.

3. Chanussot, L.; Das, A.; Goyal, S.; *et al.*  
   **Open Catalyst 2020 (OC20) Dataset and Community Challenges.**  
   *ACS Catalysis* **2021**, *11*, 6059–6072.
